# Integração da Gold de Indicadores à Base Analítica

Este notebook tem como objetivo avaliar e integrar informações provenientes
da `GOLD_INDICADORES`, construída durante a Fase 2 do Tech Challenge, à base
analítica enriquecida com dados do Atlas do Desenvolvimento Humano, Censo
Escolar e Fundeb.

A base de alunos utilizada nesta fase contém informações referentes à
avaliação de 2025. Para preservar a coerência temporal da análise e reduzir
o risco de *data leakage*, serão investigados os indicadores disponíveis
para 2024, utilizando-os como informações de contexto anteriores ao
resultado individual observado em 2025.

A análise será conduzida de forma incremental, contemplando o carregamento
e a auditoria da Gold de indicadores, o recorte temporal de 2024, a
investigação das variáveis potencialmente relevantes, a seleção dos
indicadores e, posteriormente, a preparação e validação da integração com
a base de alunos.

In [0]:
# Objetivo:
#
# Carregar a Gold de indicadores construída
# na Fase 2 do Tech Challenge.
#
# Justificativa:
#
# A base contém indicadores analíticos produzidos
# anteriormente no projeto e será avaliada como
# fonte de informações contextuais para enriquecer
# a base de alunos utilizada na Fase 3.
#
# Neste primeiro momento, nenhuma seleção ou
# transformação será realizada, preservando a
# estrutura original da fonte para auditoria.
#
# Ação:
#
# Realiza a leitura da Gold de indicadores e
# verifica sua dimensão inicial.

import pandas as pd

gold_indicadores = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/indicadores/GOLD_INDICADORES.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

alunos_atlas_censo_fundeb = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo_fundeb/alunos_atlas_censo_fundeb.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

gold_indicadores.shape

In [0]:
# Objetivo:
#
# Inspecionar a estrutura da Gold de indicadores
# antes da aplicação de qualquer filtro ou seleção.
#
# Justificativa:
#
# A base possui 16.583 registros e 46 variáveis.
#
# Antes de realizar o recorte temporal de 2024,
# é necessário conhecer os nomes das variáveis,
# seus tipos de dados e a presença de valores
# ausentes na estrutura original da fonte.
#
# Ação:
#
# Exibe as informações estruturais da Gold de
# indicadores.

gold_indicadores.info()

## 2. Recorte temporal dos indicadores

A base de alunos utilizada como população analítica corresponde à avaliação
de 2025. Para preservar a precedência temporal das informações utilizadas
como contexto preditivo, a Gold de indicadores será restringida aos
registros referentes a 2024.

Esse recorte mantém a estratégia adotada na construção da base analítica,
na qual informações contextuais anteriores ao resultado individual de 2025
são utilizadas para caracterizar o ambiente educacional dos alunos.

Após o filtro, serão realizadas novas auditorias sobre a estrutura e a
granularidade do conjunto resultante antes da investigação e seleção dos
indicadores candidatos.

In [0]:
# Objetivo:
#
# Restringir a Gold de indicadores aos
# registros referentes ao ano de 2024.
#
# Justificativa:
#
# A população analítica corresponde aos alunos
# avaliados em 2025.
#
# O uso dos indicadores de 2024 preserva a
# precedência temporal das informações contextuais
# em relação ao resultado individual observado
# em 2025 e mantém a estratégia adotada nas
# integrações anteriores.
#
# A Gold original será preservada para garantir
# rastreabilidade durante as etapas de investigação
# e seleção das variáveis.
#
# Ação:
#
# Filtra os registros cujo ANO é igual a 2024
# e cria uma cópia independente para as análises
# subsequentes.

gold_indicadores_2024 = (
    gold_indicadores[
        gold_indicadores["ANO"] == 2024
    ]
    .copy()
)

gold_indicadores_2024.shape

In [0]:
# Objetivo:
#
# Validar o recorte temporal realizado na
# Gold de indicadores.
#
# Justificativa:
#
# A base foi restringida aos registros cujo
# ANO é igual a 2024, resultando em 5.516
# registros.
#
# Antes de prosseguir para a investigação dos
# indicadores, é necessário confirmar que o
# DataFrame resultante contém exclusivamente
# informações referentes ao período selecionado.
#
# Ação:
#
# Verifica os valores distintos das variáveis
# temporais presentes na Gold após o recorte.

pd.Series({
    "ANO": gold_indicadores_2024["ANO"].unique(),
    "NU_ANO_AVALIACAO": (
        gold_indicadores_2024["NU_ANO_AVALIACAO"].unique()
    )
})

In [0]:
# Objetivo:
#
# Investigar os registros de 2024 que apresentam
# NU_ANO_AVALIACAO igual a 0.
#
# Justificativa:
#
# A validação do recorte temporal confirmou que
# todos os registros possuem ANO igual a 2024,
# porém foram identificados dois valores distintos
# em NU_ANO_AVALIACAO: 2024 e 0.
#
# Antes de interpretar ou tratar essa diferença,
# é necessário quantificar sua ocorrência e
# compreender como esses registros estão
# representados na Gold de indicadores.
#
# Ação:
#
# Contabiliza os registros de 2024 segundo os
# valores encontrados em NU_ANO_AVALIACAO.

gold_indicadores_2024[
    "NU_ANO_AVALIACAO"
].value_counts(dropna=False)

## 3. Investigação e seleção dos indicadores candidatos

Após o recorte temporal da Gold de indicadores para 2024, esta etapa tem
como objetivo identificar as variáveis que podem agregar informação à base
analítica utilizada para a predição da alfabetização dos alunos avaliados
em 2025.

A investigação será concentrada na chave municipal necessária à integração
e nos indicadores com potencial valor analítico, evitando incorporar
variáveis redundantes, derivadas desnecessariamente ou incompatíveis com
a estratégia temporal adotada no projeto.

As variáveis serão inicialmente tratadas como candidatas. A definição
daquelas que efetivamente serão incorporadas ocorrerá somente após a
avaliação de seu significado, cobertura, qualidade e possível redundância.

## 3. Investigação e seleção dos indicadores candidatos

Após o recorte temporal da Gold de indicadores para 2024, esta etapa tem
como objetivo identificar as variáveis que podem agregar informação à base
analítica utilizada para a predição da alfabetização dos alunos avaliados
em 2025.

A investigação será concentrada na chave municipal necessária à integração
e nos indicadores com potencial valor analítico, evitando incorporar
variáveis redundantes, derivadas desnecessariamente ou incompatíveis com
a estratégia temporal adotada no projeto.

As variáveis serão inicialmente tratadas como candidatas. A definição
daquelas que efetivamente serão incorporadas ocorrerá somente após a
avaliação de seu significado, cobertura, qualidade e possível redundância.

In [0]:
# Objetivo:
#
# Avaliar a chave municipal disponível na
# Gold de indicadores de 2024.
#
# Justificativa:
#
# A variável CO_MUNICIPIO está presente tanto
# na Gold de indicadores quanto na base analítica
# de alunos e é candidata à chave de integração
# entre as duas fontes.
#
# Antes da seleção dos indicadores que serão
# incorporados, é necessário verificar a
# completude e a unicidade dessa chave no
# recorte temporal selecionado.
#
# Ação:
#
# Verifica a quantidade de registros, valores
# ausentes e códigos municipais distintos na
# Gold de indicadores de 2024.

pd.Series({
    "registros": len(gold_indicadores_2024),

    "codigos_municipais_ausentes": (
        gold_indicadores_2024["CO_MUNICIPIO"]
        .isna()
        .sum()
    ),

    "codigos_municipais_distintos": (
        gold_indicadores_2024["CO_MUNICIPIO"]
        .nunique()
    )
})

In [0]:
# Objetivo:
#
# Inspecionar os indicadores candidatos da
# Gold de 2024 com potencial de agregar
# informação à base analítica.
#
# Justificativa:
#
# A chave CO_MUNICIPIO foi validada como
# completa e única no recorte de 2024.
#
# A investigação das variáveis disponíveis,
# apoiada pelo dicionário dos microdados,
# indicou como principais candidatas informações
# relacionadas ao desempenho municipal anterior,
# à proficiência em Língua Portuguesa e à meta
# estabelecida para 2025.
#
# Neste momento, as variáveis possuem caráter
# exclusivamente investigativo e ainda não
# representam a seleção definitiva para o merge.
#
# Ação:
#
# Exibe uma amostra da chave municipal e dos
# indicadores candidatos para permitir a
# inspeção inicial de seus valores.

colunas_candidatas = [
    "CO_MUNICIPIO",
    "META_FINAL_2025",
    "PC_ALUNO_ALFABETIZADO_2024"
]

gold_indicadores_2024[
    colunas_candidatas
].head(10)

In [0]:
# Objetivo:
#
# Avaliar a cobertura das variáveis selecionadas
# da Gold de indicadores de 2024.
#
# Justificativa:
#
# Após a investigação das variáveis disponíveis,
# foram selecionadas META_FINAL_2025 e
# PC_ALUNO_ALFABETIZADO_2024 para enriquecer
# a base analítica.
#
# As variáveis representam, respectivamente,
# a meta municipal estabelecida para 2025 e
# o percentual de alunos alfabetizados no
# município em 2024.
#
# Antes da preparação para o merge, é necessário
# verificar a disponibilidade dessas informações
# entre os municípios presentes na Gold.
#
# Ação:
#
# Contabiliza os valores preenchidos e ausentes
# das variáveis selecionadas.

colunas_selecionadas = [
    "META_FINAL_2025",
    "PC_ALUNO_ALFABETIZADO_2024"
]

pd.DataFrame({
    "preenchidos": (
        gold_indicadores_2024[colunas_selecionadas]
        .notna()
        .sum()
    ),
    "ausentes": (
        gold_indicadores_2024[colunas_selecionadas]
        .isna()
        .sum()
    )
})

In [0]:
# Objetivo:
#
# Verificar se as ausências identificadas nas
# variáveis selecionadas ocorrem nos mesmos
# municípios da Gold de indicadores de 2024.
#
# Justificativa:
#
# META_FINAL_2025 e PC_ALUNO_ALFABETIZADO_2024
# apresentaram exatamente 164 valores ausentes.
#
# Antes de concluir a seleção das variáveis,
# é necessário verificar se essas ausências
# possuem o mesmo padrão municipal ou se
# representam lacunas distintas de cobertura.
#
# Ação:
#
# Compara as máscaras de valores ausentes das
# duas variáveis selecionadas e contabiliza
# suas ocorrências conjuntas e exclusivas.

ausente_meta = (
    gold_indicadores_2024["META_FINAL_2025"].isna()
)

ausente_alfabetizacao = (
    gold_indicadores_2024[
        "PC_ALUNO_ALFABETIZADO_2024"
    ].isna()
)

pd.Series({
    "ausentes_em_ambas": (
        ausente_meta & ausente_alfabetizacao
    ).sum(),

    "ausente_apenas_meta": (
        ausente_meta & ~ausente_alfabetizacao
    ).sum(),

    "ausente_apenas_alfabetizacao": (
        ~ausente_meta & ausente_alfabetizacao
    ).sum()
})

### Seleção final dos indicadores

Após a investigação das variáveis disponíveis na Gold de indicadores de
2024, foram selecionadas duas informações para enriquecimento da base
analítica:

- `META_FINAL_2025`: representa a meta municipal estabelecida para o ano
  de 2025, correspondente ao período da avaliação dos alunos utilizados
  como população analítica.

- `PC_ALUNO_ALFABETIZADO_2024`: representa o percentual municipal de
  alunos alfabetizados em 2024, fornecendo uma medida histórica do
  desempenho educacional imediatamente anterior ao período analisado.

A seleção priorizou parcimônia e complementaridade em relação às variáveis
já incorporadas nas etapas anteriores de enriquecimento da base.

Dos 5.516 municípios presentes no recorte de 2024, 5.352 possuem valores
preenchidos para os dois indicadores selecionados. As 164 ausências
ocorrem simultaneamente nas duas variáveis, caracterizando um mesmo
conjunto de municípios sem cobertura para esses indicadores.

Esses registros serão preservados durante a integração, mantendo a
população original de alunos e permitindo que os valores ausentes sejam
tratados posteriormente nas etapas apropriadas do projeto.

## 4. Preparação para a integração

Após a investigação da Gold de indicadores de 2024, foram selecionadas
duas variáveis para o enriquecimento da base analítica:
`META_FINAL_2025` e `PC_ALUNO_ALFABETIZADO_2024`.

Nesta etapa, será construída uma estrutura auxiliar contendo apenas os
indicadores selecionados e a chave `CO_MUNICIPIO`, utilizada para o
relacionamento com a base de alunos.

Antes da integração, serão realizadas validações sobre a estrutura da
tabela auxiliar e a compatibilidade da chave entre as duas bases,
preservando o padrão de controle adotado nas integrações anteriores.

In [0]:
# Objetivo:
#
# Preparar a Gold de indicadores para a
# integração com a base analítica de alunos.
#
# Justificativa:
#
# A etapa de investigação definiu
# META_FINAL_2025 e PC_ALUNO_ALFABETIZADO_2024
# como as variáveis que serão incorporadas
# à base analítica.
#
# Como CO_MUNICIPIO será utilizado como chave
# de relacionamento, somente essa identificação
# e os indicadores selecionados são necessários
# para as próximas etapas.
#
# Ação:
#
# Cria uma cópia contendo apenas a chave
# municipal e as duas variáveis selecionadas
# para a integração.

indicadores_merge = gold_indicadores_2024[
    [
        "CO_MUNICIPIO",
        "META_FINAL_2025",
        "PC_ALUNO_ALFABETIZADO_2024"
    ]
].copy()

indicadores_merge.shape

In [0]:
# Objetivo:
#
# Validar a compatibilidade da chave municipal
# entre a base analítica de alunos e a tabela
# de indicadores preparada para integração.
#
# Justificativa:
#
# CO_MUNICIPIO será utilizada como chave do
# merge entre as duas bases.
#
# Antes de avaliar a correspondência entre os
# municípios, é necessário confirmar que a
# variável possui tipos de dados compatíveis,
# evitando falhas ou correspondências incorretas
# durante a integração.
#
# Ação:
#
# Verifica o tipo de dado de CO_MUNICIPIO
# nas duas bases que participarão do merge.

pd.Series({
    "alunos": alunos_atlas_censo_fundeb["CO_MUNICIPIO"].dtype,
    "indicadores": indicadores_merge["CO_MUNICIPIO"].dtype
})

In [0]:
# Objetivo:
#
# Avaliar a correspondência dos códigos municipais
# entre a base analítica de alunos e a Gold de
# indicadores preparada para integração.
#
# Justificativa:
#
# A base de alunos possui registros sem
# CO_MUNICIPIO, já identificados e documentados
# nas etapas anteriores do projeto.
#
# Para avaliar corretamente a cobertura da chave,
# a comparação deve considerar apenas os códigos
# municipais efetivamente disponíveis, sem alterar
# ou imputar os valores ausentes da base de alunos.
#
# A análise permite antecipar os municípios que
# possuem ou não correspondência antes da execução
# do merge.
#
# Ação:
#
# Cria conjuntos com os códigos municipais
# disponíveis nas duas bases e contabiliza
# municípios em comum e sem correspondência
# em cada uma delas.

municipios_alunos = set(
    alunos_atlas_censo_fundeb["CO_MUNICIPIO"]
    .dropna()
    .astype("int64")
    .unique()
)

municipios_indicadores = set(
    indicadores_merge["CO_MUNICIPIO"]
    .unique()
)

pd.Series({
    "municipios_alunos": len(municipios_alunos),
    "municipios_indicadores": len(municipios_indicadores),
    "municipios_em_comum": len(
        municipios_alunos & municipios_indicadores
    ),
    "alunos_sem_correspondencia_indicadores": len(
        municipios_alunos - municipios_indicadores
    ),
    "indicadores_sem_correspondencia_alunos": len(
        municipios_indicadores - municipios_alunos
    )
})

In [0]:
# Objetivo:
#
# Identificar os municípios presentes na base
# de alunos que não possuem correspondência na
# Gold de indicadores de 2024.
#
# Justificativa:
#
# A auditoria prévia das chaves identificou
# 48 códigos municipais presentes na base de
# alunos e ausentes na Gold de indicadores.
#
# Antes da integração, é necessário identificar
# esses municípios e avaliar se as diferenças
# decorrem de características conhecidas das
# fontes ou de possíveis inconsistências na
# chave de relacionamento.
#
# Ação:
#
# Seleciona os registros correspondentes aos
# códigos municipais sem correspondência na
# Gold de indicadores e apresenta município,
# UF e quantidade de alunos.

municipios_sem_indicadores = (
    alunos_atlas_censo_fundeb[
        alunos_atlas_censo_fundeb["CO_MUNICIPIO"]
        .isin(
            municipios_alunos - municipios_indicadores
        )
    ]
    .groupby(
        ["CO_MUNICIPIO", "NO_MUNICIPIO", "SG_UF"],
        dropna=False
    )
    .size()
    .reset_index(name="quantidade_alunos")
    .sort_values(
        "quantidade_alunos",
        ascending=False
    )
)

municipios_sem_indicadores

In [0]:
# Objetivo:
#
# Identificar os municípios presentes na Gold
# de indicadores de 2024 que não possuem
# correspondência na base analítica de alunos.
#
# Justificativa:
#
# A auditoria das chaves identificou 8 códigos
# municipais presentes na Gold de indicadores
# e ausentes na base de alunos.
#
# A identificação desses municípios complementa
# a análise de cobertura das duas fontes e
# permite compreender as diferenças existentes
# antes da execução do merge.
#
# Ação:
#
# Seleciona os registros da Gold correspondentes
# aos códigos municipais sem correspondência
# na base analítica de alunos.

indicadores_merge[
    indicadores_merge["CO_MUNICIPIO"].isin(
        municipios_indicadores - municipios_alunos
    )
]

In [0]:
# Objetivo:
#
# Avaliar a disponibilidade efetiva dos
# indicadores selecionados entre os municípios
# da base analítica que possuem correspondência
# na Gold de 2024.
#
# Justificativa:
#
# A correspondência entre CO_MUNICIPIO nas duas
# bases não garante que os indicadores selecionados
# estejam preenchidos.
#
# Foram identificados 164 municípios na Gold
# com ausência simultânea de META_FINAL_2025 e
# PC_ALUNO_ALFABETIZADO_2024.
#
# Antes do merge, é necessário verificar quantos
# desses municípios fazem parte da população
# analítica de alunos.
#
# Ação:
#
# Identifica os municípios presentes nas duas
# bases que não possuem valores para os dois
# indicadores selecionados.

municipios_sem_valores_indicadores = (
    indicadores_merge[
        indicadores_merge["CO_MUNICIPIO"].isin(
            municipios_alunos
        )
        &
        indicadores_merge["META_FINAL_2025"].isna()
        &
        indicadores_merge[
            "PC_ALUNO_ALFABETIZADO_2024"
        ].isna()
    ]
)

municipios_sem_valores_indicadores.shape

In [0]:
# Objetivo:
#
# Quantificar os alunos que não receberão valores
# dos indicadores selecionados após a integração.
#
# Justificativa:
#
# A auditoria pré-merge identificou três situações
# distintas que podem resultar em valores ausentes:
# alunos sem CO_MUNICIPIO, municípios sem
# correspondência na Gold e municípios encontrados
# na Gold cujos indicadores estão ausentes.
#
# Antes da execução do merge, é necessário
# quantificar o impacto dessas situações na
# população de alunos para permitir posteriormente
# a reconciliação dos valores ausentes.
#
# Ação:
#
# Contabiliza os alunos pertencentes a cada uma
# das situações identificadas antes da integração.

sem_codigo_municipal = (
    alunos_atlas_censo_fundeb["CO_MUNICIPIO"]
    .isna()
    .sum()
)

sem_correspondencia_indicadores = (
    alunos_atlas_censo_fundeb["CO_MUNICIPIO"]
    .isin(
        municipios_alunos - municipios_indicadores
    )
    .sum()
)

sem_valores_indicadores = (
    alunos_atlas_censo_fundeb["CO_MUNICIPIO"]
    .isin(
        municipios_sem_valores_indicadores[
            "CO_MUNICIPIO"
        ]
    )
    .sum()
)

pd.Series({
    "sem_codigo_municipal": sem_codigo_municipal,
    "sem_correspondencia_indicadores": (
        sem_correspondencia_indicadores
    ),
    "sem_valores_indicadores": (
        sem_valores_indicadores
    )
})

### Resultado da preparação para a integração

A tabela auxiliar foi preparada com 5.516 registros municipais e as duas
variáveis selecionadas para enriquecimento da base analítica.

A auditoria das chaves identificou 5.508 municípios em comum entre as
fontes, além de 48 municípios presentes na base de alunos sem
correspondência na Gold de indicadores e 8 municípios presentes somente
na Gold.

Também foram identificados 163 municípios da população analítica que,
apesar de possuírem correspondência na Gold, não apresentam valores para
os dois indicadores selecionados.

Considerando os 510 alunos sem código municipal, os 14.505 alunos
pertencentes a municípios sem correspondência e os 70.256 alunos
pertencentes a municípios com indicadores ausentes, são esperados 85.271
valores ausentes em cada uma das duas variáveis após a integração.

Esses registros serão preservados por meio de uma integração à esquerda,
mantendo integralmente a população original de alunos.

## 5. Integração da Gold de indicadores à base analítica

Após a preparação e validação das chaves, a Gold de indicadores de 2024
será integrada à base analítica de alunos.

A integração utilizará `CO_MUNICIPIO` como chave de relacionamento e
preservará integralmente a população de alunos por meio de um `left join`.

A cardinalidade esperada é `many_to_one`, uma vez que a base analítica
possui múltiplos alunos por município, enquanto a tabela de indicadores
possui um único registro para cada código municipal.

Serão incorporadas as variáveis `META_FINAL_2025` e
`PC_ALUNO_ALFABETIZADO_2024`. O indicador `_merge` será mantido
temporariamente para permitir a auditoria da correspondência entre as
fontes antes da consolidação da base.

In [0]:
# Objetivo:
#
# Integrar os indicadores municipais selecionados
# à base analítica de alunos.
#
# Justificativa:
#
# A etapa de preparação confirmou que
# CO_MUNICIPIO pode ser utilizada como chave
# de relacionamento entre as duas fontes.
#
# A tabela de indicadores possui um único
# registro por município, enquanto a base
# analítica contém múltiplos alunos associados
# ao mesmo código municipal.
#
# Dessa forma, a cardinalidade esperada para
# a integração é many_to_one.
#
# O left join preserva integralmente a população
# original de alunos, inclusive os registros sem
# correspondência ou sem código municipal.
#
# Ação:
#
# Realiza a integração pela chave CO_MUNICIPIO,
# valida a cardinalidade esperada e mantém o
# indicador de origem para auditoria posterior.

alunos_atlas_censo_fundeb_indicadores = (
    alunos_atlas_censo_fundeb.merge(
        indicadores_merge,
        on="CO_MUNICIPIO",
        how="left",
        validate="many_to_one",
        indicator=True
    )
)

alunos_atlas_censo_fundeb_indicadores.shape

In [0]:
# Objetivo:
#
# Validar a correspondência dos registros após
# a integração da Gold de indicadores à base
# analítica de alunos.
#
# Justificativa:
#
# A auditoria pré-merge identificou 510 alunos
# sem CO_MUNICIPIO e 14.505 alunos pertencentes
# a municípios sem correspondência na Gold de
# indicadores.
#
# Dessa forma, são esperados 15.015 registros
# classificados como left_only após a integração.
#
# Os alunos pertencentes a municípios encontrados
# na Gold, mas com indicadores ausentes, devem ser
# classificados como both, pois possuem
# correspondência pela chave municipal.
#
# Ação:
#
# Contabiliza os registros segundo o indicador
# de correspondência gerado pelo merge.

alunos_atlas_censo_fundeb_indicadores[
    "_merge"
].value_counts()

In [0]:
# Objetivo:
#
# Validar a quantidade de valores ausentes nas
# variáveis incorporadas após a integração.
#
# Justificativa:
#
# A auditoria realizada antes do merge identificou
# três situações capazes de gerar valores ausentes
# nos indicadores: alunos sem CO_MUNICIPIO, alunos
# pertencentes a municípios sem correspondência na
# Gold e alunos de municípios encontrados na Gold
# cujos indicadores não possuem valores preenchidos.
#
# A soma dessas situações resultou em uma expectativa
# de 85.271 valores ausentes para cada uma das duas
# variáveis incorporadas.
#
# A comparação entre os valores esperados e observados
# permite verificar se as ausências após o merge são
# integralmente explicadas pelas condições previamente
# identificadas.
#
# Ação:
#
# Compara a quantidade observada de valores ausentes
# nas variáveis incorporadas com a quantidade esperada
# a partir da auditoria pré-merge.

ausentes_esperados = (
    sem_codigo_municipal
    + sem_correspondencia_indicadores
    + sem_valores_indicadores
)

pd.DataFrame({
    "ausentes": [
        alunos_atlas_censo_fundeb_indicadores[
            "META_FINAL_2025"
        ].isna().sum(),

        alunos_atlas_censo_fundeb_indicadores[
            "PC_ALUNO_ALFABETIZADO_2024"
        ].isna().sum()
    ],
    "esperado": [
        ausentes_esperados,
        ausentes_esperados
    ]
},
    index=[
        "META_FINAL_2025",
        "PC_ALUNO_ALFABETIZADO_2024"
    ]
).assign(
    conforme=lambda df: df["ausentes"] == df["esperado"]
)

In [0]:
# Objetivo:
#
# Consolidar a base analítica após a integração
# com a Gold de indicadores de 2024.
#
# Justificativa:
#
# As validações pós-merge confirmaram a
# preservação da população original de alunos,
# a cardinalidade esperada da integração e a
# correspondência entre as ausências observadas
# e aquelas previstas na auditoria pré-merge.
#
# A variável auxiliar _merge já cumpriu sua
# finalidade de auditoria e não possui valor
# analítico para as próximas etapas do projeto.
#
# Ação:
#
# Remove a variável auxiliar _merge e verifica
# a dimensão da base consolidada.

alunos_atlas_censo_fundeb_indicadores = (
    alunos_atlas_censo_fundeb_indicadores
    .drop(columns="_merge")
)

alunos_atlas_censo_fundeb_indicadores.shape

In [0]:
# Objetivo:
#
# Validar a estrutura final da base analítica
# após a integração com a Gold de indicadores.
#
# Justificativa:
#
# A integração preservou os 1.966.605 registros
# da população original e acrescentou as duas
# variáveis selecionadas, resultando em uma base
# com 41 colunas.
#
# Antes da persistência, é necessário verificar
# a estrutura consolidada da base, incluindo os
# tipos de dados e a presença das novas variáveis
# incorporadas.
#
# Ação:
#
# Exibe as informações estruturais da base
# analítica consolidada após a integração.

alunos_atlas_censo_fundeb_indicadores.info()

## 6. Persistência da base analítica enriquecida

Após a integração e validação dos indicadores selecionados, a base
analítica enriquecida será persistida para utilização nas próximas
etapas do projeto.

O arquivo consolidado preserva os 1.966.605 registros da população
original e passa a conter 41 variáveis, incorporando
`META_FINAL_2025` e `PC_ALUNO_ALFABETIZADO_2024` às informações
provenientes dos enriquecimentos realizados anteriormente.

Após a gravação, o arquivo será relido para confirmar sua integridade
estrutural antes de sua utilização nas próximas etapas.

In [0]:
# Objetivo:
#
# Persistir a base analítica após a integração
# com a Gold de indicadores de 2024.
#
# Justificativa:
#
# A integração foi validada quanto à cardinalidade,
# preservação da população original, correspondência
# das chaves e completude das variáveis incorporadas.
#
# A persistência consolida o resultado desta etapa
# e disponibiliza a base enriquecida para utilização
# nas próximas fases do projeto.
#
# Ação:
#
# Salva a base analítica consolidada em formato CSV,
# utilizando o padrão de separador e codificação
# adotado nas etapas anteriores.

alunos_atlas_censo_fundeb_indicadores.to_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo_fundeb_indicadores/alunos_atlas_censo_fundeb_indicadores.csv",
    sep=";",
    encoding="utf-8-sig",
    index=False
)

In [0]:
# Objetivo:
#
# Validar o arquivo persistido após a
# integração com a Gold de indicadores de 2024.
#
# Justificativa:
#
# A releitura do arquivo permite confirmar
# que a base foi gravada corretamente e
# preservou sua dimensão antes de ser utilizada
# nas próximas etapas do projeto.
#
# Ação:
#
# Realiza uma leitura de controle do arquivo
# persistido e verifica sua dimensão.

df_validacao = pd.read_csv(
    "/Volumes/workspace/default/vol_trio_drive/projetos/fiap/tech_challenge_fase2/gold/alunos_atlas_censo_fundeb_indicadores/alunos_atlas_censo_fundeb_indicadores.csv",
    sep=";",
    encoding="utf-8-sig",
    low_memory=False
)

df_validacao.shape